# Component D — IEMOCAP training (real continuous valence/arousal)

**The domain-gap fix.** Trains the V/A head on IEMOCAP's **real human ratings**
(EmoVal / EmoAct, 1-5) from the open HuggingFace mirror `AbstractTTS/IEMOCAP` -
instead of the per-emotion lookup table. This is the data meant to sharpen
valence/arousal on real voices. The 1-5 ratings are rescaled to [-1, 1].

**Before running:** Runtime → Change runtime type → **T4 GPU**. No Kaggle token
needed (HF is open). Features + model cache to Drive so a dropped session is safe.

*If the dataset ever asks for access, run `from huggingface_hub import login; login()`
with a free HF token first.* Cite **Busso et al. (2008), IEMOCAP** in the report.

In [ ]:
# 1. Mount Drive (cache features + model)
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/cognivoice'
os.makedirs(DRIVE, exist_ok=True)
print('Drive mounted at', DRIVE)

In [ ]:
# 2. Get the code + install dependencies (~3 min)
%cd /content
!rm -rf cognivoice-component-d
!git clone -q https://github.com/Prathikesh/cognivoice-component-d.git
%cd cognivoice-component-d
!pip install -q funasr modelscope librosa soundfile praat-parselmouth scipy scikit-learn tqdm datasets
print('setup done')

In [ ]:
# 3. Download the HF IEMOCAP mirror -> wavs + metadata with REAL V/A
#    (EmoVal/EmoAct rescaled 1-5 -> [-1,1]; speaker-independent split)
import os
os.makedirs('data', exist_ok=True); os.makedirs('models', exist_ok=True)
!python scripts/build_iemocap_hf.py --out-wav data/wav/iemocap --out data/metadata_iemocap.csv

In [ ]:
# 4. FEATURE EXTRACTION on IEMOCAP (~30-60 min on T4). Same frozen encoder +
# prosody + shared preprocessing as everything else. Cached to Drive.
OUT = 'data/features_iemocap.npz'
if os.path.exists(f'{DRIVE}/features_iemocap.npz'):
    print('features cached in Drive - skipping extraction')
    !cp "{DRIVE}/features_iemocap.npz" {OUT}
else:
    !python scripts/extract_features.py --metadata data/metadata_iemocap.csv --out {OUT} --encoder plus_large
    !cp {OUT} "{DRIVE}/features_iemocap.npz"
    print('features cached to Drive')

In [ ]:
# 5. TRAIN on IEMOCAP's real V/A (fast). Saved SEPARATELY so you can compare
# it to the acted model on the real-voice acid test.
!python scripts/train_fusion.py --features {OUT} --out models/fusion_iemocap.pt

In [ ]:
# 6. COMBINED model = IEMOCAP (real V/A) + acted (valence range). Often the
# best of both. Uses the cached features from BOTH runs - no re-extraction.
import os
for f in ['features_iemocap.npz', 'features_actedreal.npz']:
    if os.path.exists(f'{DRIVE}/{f}') and not os.path.exists(f'data/{f}'):
        !cp "{DRIVE}/{f}" data/{f}
if os.path.exists('data/features_actedreal.npz'):
    !python scripts/train_fusion.py --features data/features_iemocap.npz data/features_actedreal.npz --out models/fusion_combined.pt
    !cp models/fusion_combined.pt "{DRIVE}/fusion_combined.pt"
    print('saved fusion_combined.pt to Drive')
else:
    print('features_actedreal.npz not on Drive yet - run colab_train.ipynb once to create it, then re-run this cell')

In [ ]:
# 7. Save model + history to Drive
!cp models/fusion_iemocap.pt "{DRIVE}/fusion_iemocap.pt"
!cp models/fusion_iemocap.history.json "{DRIVE}/fusion_iemocap.history.json" 2>/dev/null || true
print('saved fusion_iemocap.pt to Drive -> download into models/ on your Mac to test')